In [1]:
import pandas as pd

# 1. Carga de Bases Normalizadas
df_mayor = pd.read_csv("../data/raw/libro_mayor.csv")
df_banco = pd.read_csv("../data/raw/extracto_bancario.csv")

df_mayor['fecha_contable'] = pd.to_datetime(df_mayor['fecha_contable'])
df_banco['fecha_banco'] = pd.to_datetime(df_banco['fecha_banco'])

# Normalizamos el espejo bancario
df_banco['monto_normalizado'] = df_banco['monto'] * -1

# --- NIVEL 1: MATCH EXACTO POR COMPROBANTE Y MONTO ---
match_exacto = pd.merge(
    df_mayor,
    df_banco,
    on=['ref_comprobante'],
    suffixes=('_mayor', '_banco')
)

# Filtramos coincidencias exactas de monto
match_exacto['dias_clearing'] = (match_exacto['fecha_banco'] - match_exacto['fecha_contable']).dt.days
match_exacto['diferencia_monto'] = match_exacto['monto_mayor'] - match_exacto['monto_normalizado']

# Clasificación dentro de las 72 hs (0 a 3 días) sin desvío de monto
conciliados_n1 = match_exacto[
    (match_exacto['diferencia_monto'] == 0) &
    (match_exacto['dias_clearing'] >= 0) &
    (match_exacto['dias_clearing'] <= 3)
].copy()

print(f"✅ Nivel 1 (Conciliados a término <= 72hs): {len(conciliados_n1)} partidas")

✅ Nivel 1 (Conciliados a término <= 72hs): 802 partidas


In [2]:
# --- NIVEL 2: CLASIFICACIÓN DE AUDITORÍA INTEGRAL ---
# Identificamos el total de relaciones cruzadas
todas_partidas = pd.merge(
    df_mayor,
    df_banco,
    on='ref_comprobante',
    how='outer',
    suffixes=('_mayor', '_banco')
)

todas_partidas['dias_clearing'] = (todas_partidas['fecha_banco'] - todas_partidas['fecha_contable']).dt.days
todas_partidas['monto_banco_normalizado'] = todas_partidas['monto_banco'] * -1

def clasificar_partida(row):
    if pd.notna(row['ref_comprobante']):
        if pd.notna(row['fecha_contable']) and pd.notna(row['fecha_banco']):
            if row['monto_mayor'] == row['monto_banco_normalizado']:
                if 0 <= row['dias_clearing'] <= 3:
                    return 'CONCILIADO_DENTRO_DE_CLEARING'
                elif row['dias_clearing'] < 0:
                    return 'ALERTA_BANCO_ANTERIOR_A_MAYOR'
                else:
                    return 'CONCILIADO_FUERA_DE_TERMINO'
            else:
                return 'DISCREPANCIA_DE_MONTO'
        elif pd.notna(row['fecha_contable']) and pd.isna(row['fecha_banco']):
            return 'PENDIENTE_ACREDITACION_BANCO'
        elif pd.isna(row['fecha_contable']) and pd.notna(row['fecha_banco']):
            return 'OMISION_REGISTRO_MAYOR'
    return 'REVISION_MANUAL'

todas_partidas['estado_audit'] = todas_partidas.apply(clasificar_partida, axis=1)

print("📊 Resumen del Motor de Auditoría:")
print(todas_partidas['estado_audit'].value_counts())

📊 Resumen del Motor de Auditoría:
estado_audit
CONCILIADO_DENTRO_DE_CLEARING    802
CONCILIADO_FUERA_DE_TERMINO      197
DISCREPANCIA_DE_MONTO              1
OMISION_REGISTRO_MAYOR             1
Name: count, dtype: int64


In [3]:
import os

ruta_processed = "../data/processed"
os.makedirs(ruta_processed, exist_ok=True)

# Guardar evidencia final conciliada y partidas pendientes
todas_partidas.to_csv(f"{ruta_processed}/conciliados_final.csv", index=False)

# Partidas para revisión contable en Excel
pendientes_mayor = todas_partidas[todas_partidas['estado_audit'] == 'PENDIENTE_ACREDITACION_BANCO']
pendientes_banco = todas_partidas[todas_partidas['estado_audit'] == 'OMISION_REGISTRO_MAYOR']

with pd.ExcelWriter(f"{ruta_processed}/Partidas_Pendientes_Marzo.xlsx", engine='openpyxl') as writer:
    pendientes_mayor.to_excel(writer, sheet_name='Revisar_en_Mayor', index=False)
    pendientes_banco.to_excel(writer, sheet_name='Revisar_en_Banco', index=False)

print("✅ Evidencia exportada a data/processed/ exitosamente.")

✅ Evidencia exportada a data/processed/ exitosamente.
